# LeetCode 445 — Add Two Numbers II bằng Python

Link đề: https://leetcode.com/problems/add-two-numbers-ii/ · Độ khó: **Medium** · Chủ đề: Linked List, Stack, Math

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

Ghi chú: đề và ví dụ được viết lại theo hiểu biết về đề gốc; trong notebook ta **tự dựng `ListNode`** để chạy được ngoài LeetCode. Mọi đáp án đã được kiểm chứng bằng code (đối chiếu với phép cộng số nguyên của Python), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai danh sách liên kết đơn biểu diễn hai số nguyên không âm, **chữ số hàng cao nhất nằm ở đầu danh sách**; hãy cộng chúng và trả về tổng cũng dưới dạng danh sách liên kết (chữ số cao nhất ở đầu). Câu hỏi nâng cao: làm **không đảo ngược** danh sách đầu vào.
2. Nắm các công cụ Python: lớp `ListNode`, duyệt danh sách liên kết, `list` dùng làm **ngăn xếp (stack)** (`append`/`pop`), `divmod` để tách chữ số và số nhớ, chèn nút vào **đầu** danh sách, đảo ngược danh sách, và đệ quy.
3. Đi từ **đổi cả hai danh sách thành số nguyên** → **đảo ngược cả hai rồi cộng từ đầu** → **hai ngăn xếp + chèn đầu** (đáp án cho câu hỏi nâng cao) → **căn hàng theo độ dài rồi đệ quy**.
4. Hiểu mấu chốt: khi cộng tay ta bắt đầu từ chữ số **thấp nhất** (cuối), nhưng danh sách chỉ cho đi từ **cao** (đầu) đến thấp — cần một cấu trúc **đảo thứ tự** (ngăn xếp, đệ quy, hoặc đảo danh sách); và kết quả dựng bằng cách **chèn vào đầu** để chữ số cao nằm đầu.
5. Tránh các lỗi kinh điển: quên **số nhớ cuối cùng**, bỏ sót các chữ số của số dài hơn, dựng kết quả bằng cách nối vào đuôi (bị ngược), và cộng các chữ số **căn lề trái** thay vì căn lề phải.

## 2. Đề bài & ví dụ

Cho hai danh sách liên kết đơn **khác rỗng** `l1` và `l2` biểu diễn hai số nguyên **không âm**. Chữ số **cao nhất** nằm ở **đầu** danh sách, mỗi nút chứa **một chữ số**. Hãy cộng hai số và trả về tổng dưới dạng danh sách liên kết.

Có thể giả sử hai số không có chữ số `0` ở đầu, trừ chính số `0`.

**Ví dụ 1**

- Input: `l1 = [7,2,4,3]`, `l2 = [5,6,4]`
- Output: `[7,8,0,7]`
- Giải thích: `7243 + 564 = 7807`.

**Ví dụ 2**

- Input: `l1 = [2,4,3]`, `l2 = [5,6,4]`
- Output: `[8,0,7]`
- Giải thích: `243 + 564 = 807`.

**Ví dụ 3**

- Input: `l1 = [0]`, `l2 = [0]`
- Output: `[0]`

**Ràng buộc:**

- Số nút của mỗi danh sách nằm trong `[1, 100]`.
- `0 <= Node.val <= 9`
- Danh sách biểu diễn một số không có chữ số `0` dẫn đầu (trừ số `0`).

**Câu hỏi nâng cao (follow-up):** Bạn có giải được mà **không đảo ngược** danh sách đầu vào không?

In [1]:
from typing import Optional, List
import random
import time
import sys


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build(arr):
    # dựng danh sách liên kết từ list Python: [7, 2, 4, 3] -> 7 -> 2 -> 4 -> 3
    dummy = ListNode()
    tail = dummy
    for x in arr:
        tail.next = ListNode(x)
        tail = tail.next
    return dummy.next


def to_list(node):
    out = []
    while node:
        out.append(node.val)
        node = node.next
    return out


examples = [
    ([7, 2, 4, 3], [5, 6, 4], [7, 8, 0, 7]),
    ([2, 4, 3], [5, 6, 4], [8, 0, 7]),
    ([0], [0], [0]),
]
for a, b, expected in examples:
    print(to_list(build(a)), "+", to_list(build(b)), "-> kỳ vọng", expected)

[7, 2, 4, 3] + [5, 6, 4] -> kỳ vọng [7, 8, 0, 7]
[2, 4, 3] + [5, 6, 4] -> kỳ vọng [8, 0, 7]
[0] + [0] -> kỳ vọng [0]


## 3. Kiến thức nền

### 3.1. Nút danh sách liên kết `ListNode`

Mỗi nút gồm hai phần: `val` (giá trị — ở bài này là một chữ số) và `next` (tham chiếu đến nút kế tiếp, hoặc `None` nếu là nút cuối). Danh sách được "cầm" bằng nút **đầu** (`head`). Hai hàm `build`/`to_list` ở trên giúp ta đổi qua lại giữa `list` thường và danh sách liên kết để dễ in/ kiểm tra.

In [2]:
a = ListNode(7)
a.next = ListNode(2)
a.next.next = ListNode(4)
print(a.val, a.next.val, a.next.next.val, a.next.next.next)    # 7 2 4 None
print(to_list(a))

7 2 4 None
[7, 2, 4]


### 3.2. Duyệt danh sách và đếm độ dài

Ta đi từ `head`, mỗi lần `node = node.next` cho đến khi `node` là `None`. Danh sách liên kết **không có** chỉ số: không thể nhảy thẳng đến nút thứ `k`, cũng không thể đi **lùi**. Chính điều đó khiến bài này khó: ta muốn cộng từ chữ số thấp nhất (cuối), nhưng chỉ đi được từ đầu.

In [3]:
def length(node):
    n = 0
    while node:
        n += 1
        node = node.next
    return n


print(length(build([7, 2, 4, 3])), length(build([5, 6, 4])), length(None))

4 3 0


### 3.3. Ngăn xếp (stack) bằng `list`: vào sau, ra trước

`stack.append(x)` đẩy vào đỉnh, `stack.pop()` lấy ra từ đỉnh (**vào sau, ra trước** — LIFO). Nếu ta đẩy lần lượt các chữ số `7, 2, 4, 3` rồi lấy ra, ta nhận `3, 4, 2, 7` — **đúng thứ tự từ thấp lên cao**. Đây là công cụ đảo thứ tự mà ta cần.

In [4]:
stack = []
for d in [7, 2, 4, 3]:
    stack.append(d)
print(stack)
print([stack.pop() for _ in range(len(stack))])      # [3, 4, 2, 7]

[7, 2, 4, 3]
[3, 4, 2, 7]


### 3.4. Phép cộng có nhớ và `divmod`

Cộng hai chữ số `x`, `y` cùng số nhớ `carry` (0 hoặc 1): `total = x + y + carry`. Chữ số ghi xuống là `total % 10`, số nhớ mới là `total // 10`. `divmod(total, 10)` trả về cả hai: `(total // 10, total % 10)`.

In [5]:
print(divmod(3 + 4, 10))        # (0, 7): 3 + 4 = 7, nhớ 0
print(divmod(7 + 5 + 1, 10))    # (1, 3): 7 + 5 + nhớ 1 = 13, ghi 3 nhớ 1
print(divmod(9 + 9 + 1, 10))    # (1, 9): lớn nhất có thể: 19

(0, 7)
(1, 3)
(1, 9)


### 3.5. Chèn vào **đầu** danh sách

Để dựng số kết quả có chữ số cao ở đầu trong khi ta tính từ chữ số thấp, ta **chèn mỗi chữ số mới vào đầu** danh sách đang có: `node = ListNode(d); node.next = head; head = node` (hay gọn: `head = ListNode(d, head)`). Thao tác `O(1)`.

In [6]:
head = None
for d in [7, 0, 8, 7][::-1]:        # giả sử tính được 7 (thấp nhất), rồi 0, rồi 8, rồi 7
    head = ListNode(d, head)
    print("sau khi chèn", d, ":", to_list(head))

sau khi chèn 7 : [7]
sau khi chèn 8 : [8, 7]
sau khi chèn 0 : [0, 8, 7]
sau khi chèn 7 : [7, 0, 8, 7]


### 3.6. Đảo ngược danh sách liên kết (iterative)

Duyệt `cur` từ đầu, giữ `prev`; mỗi bước đổi `cur.next` trỏ ngược về `prev`: `nxt = cur.next; cur.next = prev; prev = cur; cur = nxt`. Kết quả là `prev` — đầu mới. Việc này **thay đổi** danh sách đầu vào (bị đảo), đó là điều câu hỏi nâng cao muốn tránh.

In [7]:
def reverse_list(head):
    prev = None
    cur = head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev


h = build([7, 2, 4, 3])
r = reverse_list(h)
print(to_list(r))                # [3, 4, 2, 7]
print(to_list(h))                # [7]: nút 7 cũ giờ là đuôi, h không còn là "đầu cả danh sách"

[3, 4, 2, 7]
[7]


### 3.7. Đệ quy: "ngăn xếp ngầm" của máy tính

Một hàm đệ quy gọi chính nó: các lời gọi xếp chồng lên nhau (ngăn xếp lời gọi), và khi trả về thì **lời gọi sau cùng xong trước**. Ta có thể "đi xuống" đến cuối danh sách rồi, trên đường **quay về**, xử lý chữ số từ thấp lên cao — không cần tự tạo ngăn xếp.

In [8]:
def print_reverse(node):
    if node is None:
        return
    print_reverse(node.next)      # đi xuống hết trước
    print(node.val, end=" ")      # rồi in trên đường quay về


print_reverse(build([7, 2, 4, 3]))    # 3 4 2 7
print()

3 4 2 7 


### 3.8. Số nguyên của Python không tràn

Python có số nguyên cỡ tùy ý: `int("9" * 100)` vẫn chính xác. Nhờ vậy ta có thể dùng nó làm **oracle** (đáp án đối chiếu). Ở các ngôn ngữ như Java/C++, số 100 chữ số **không** nằm vừa kiểu `long`, nên cách "đổi thành số nguyên" không dùng được — đó là lý do đề có thiết kế bằng danh sách liên kết.

In [9]:
x = int("9" * 100)
print(x + 1 == 10 ** 100)       # True
print(len(str(x + x)))          # 101 chữ số

True
101


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cộng `7243 + 564` bằng tay theo cột (căn **phải**):

```
  7 2 4 3
+   5 6 4
---------
```

1. Hàng đơn vị: `3 + 4 = 7`, ghi `7`, nhớ `0`.
2. Hàng chục: `4 + 6 = 10`, ghi `0`, nhớ `1`.
3. Hàng trăm: `2 + 5 + 1 = 8`, ghi `8`, nhớ `0`.
4. Hàng nghìn: `7 + 0 (hết số thứ hai) = 7`, ghi `7`.

Tổng `7807` ⇒ danh sách `[7, 8, 0, 7]`.

Ta phải đi từ phải sang trái, nhưng danh sách liên kết chỉ đi được từ trái sang phải. Ta cần **đảo thứ tự**: ngăn xếp, đệ quy, hoặc đảo danh sách.

| Cột (từ phải) | `l1` | `l2` | `carry` vào | `total` | Ghi | `carry` ra |
|---|---|---|---|---|---|---|
| 1 | 3 | 4 | 0 | 7 | 7 | 0 |
| 2 | 4 | 6 | 0 | 10 | 0 | 1 |
| 3 | 2 | 5 | 1 | 8 | 8 | 0 |
| 4 | 7 | (không có → 0) | 0 | 7 | 7 | 0 |

In [10]:
a, b = [7, 2, 4, 3], [5, 6, 4]
carry = 0
out = []
i, j = len(a) - 1, len(b) - 1          # chỉ số từ phải (ở đây ta dùng list Python có chỉ số)
while i >= 0 or j >= 0 or carry:
    x = a[i] if i >= 0 else 0
    y = b[j] if j >= 0 else 0
    carry, digit = divmod(x + y + carry, 10)
    out.append(digit)
    print(f"x={x} y={y} -> ghi {digit}, nhớ {carry}")
    i -= 1
    j -= 1
print(out[::-1])

x=3 y=4 -> ghi 7, nhớ 0
x=4 y=6 -> ghi 0, nhớ 1
x=2 y=5 -> ghi 8, nhớ 0
x=7 y=0 -> ghi 7, nhớ 0
[7, 8, 0, 7]


Với `list` Python, ta có chỉ số nên làm rất dễ. Bài toán thật sự là làm điều này với **danh sách liên kết** không có chỉ số.

## 5. Cách 1 — Brute Force / baseline: đổi thành số nguyên, cộng, dựng lại

**Ý tưởng.** Đọc từng danh sách thành một số nguyên (`num = num * 10 + digit`), cộng hai số, rồi tách các chữ số của tổng thành danh sách mới (dùng `str(total)`).

In [11]:
def to_int(node):
    num = 0
    while node:
        num = num * 10 + node.val
        node = node.next
    return num


def add_via_int(l1, l2):
    total = to_int(l1) + to_int(l2)
    return build([int(c) for c in str(total)])


for a, b, expected in examples:
    print(to_list(add_via_int(build(a), build(b))), "(kỳ vọng", expected, ")")

[7, 8, 0, 7] (kỳ vọng [7, 8, 0, 7] )
[8, 0, 7] (kỳ vọng [8, 0, 7] )
[0] (kỳ vọng [0] )


**Độ phức tạp (Cách 1).** Đọc `O(n + m)` bước, nhưng mỗi phép `num * 10 + digit` trên số lớn tốn `O(độ dài số)`, nên đọc tổng cộng cỡ `O(n²)` về mặt bit (với `n <= 100` thì không đáng kể). **Space `O(n + m)`** (số lớn + danh sách kết quả).

**Nút thắt:** chỉ dùng được ở ngôn ngữ có số nguyên cỡ tùy ý như Python; ở Java/C++ số 100 chữ số tràn kiểu `long`. Đề muốn ta **thao tác trên từng chữ số**.

## 6. Ý tưởng thuật toán / bước đột phá

Phép cộng từng cột cần đi từ chữ số thấp (cuối danh sách). Ta có ba cách "quay ngược" thứ tự:

1. **Đảo ngược** cả hai danh sách rồi cộng như bài **Add Two Numbers (I)** (chữ số thấp ở đầu), rồi đảo kết quả. Dễ nhưng **sửa** đầu vào.
2. **Ngăn xếp**: đẩy các chữ số vào hai ngăn xếp rồi `pop` từ thấp lên cao; dựng kết quả bằng **chèn vào đầu**. Không đụng vào danh sách gốc — trả lời câu hỏi nâng cao.
3. **Đệ quy** + **căn hàng**: đếm độ dài, tính chênh lệch `diff`; số dài hơn có `diff` nút đầu "không có đối tác". Đệ quy đi hết hai danh sách cùng nhịp, và trên đường quay về cộng chữ số kèm số nhớ.

Điểm chung: **số nhớ chạy từ thấp lên cao** nên ta chỉ có thể tính tổng một cột khi đã biết số nhớ từ cột bên phải. Kết quả được dựng **từ cuối ra đầu** (chèn đầu).

## 7. Các cách tiếp theo

Tổng cộng **4 cách**: (1) đổi thành số nguyên (mục 5); (2) đảo ngược hai danh sách; (3) hai ngăn xếp; (4) căn hàng + đệ quy. Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Đảo ngược cả hai rồi cộng (như Add Two Numbers I)

**Giải thích từng đoạn.**

- `a = reverse_list(l1)`, `b = reverse_list(l2)`: chữ số thấp nhất nay ở đầu.
- Duyệt song song `a`, `b`; mỗi bước `total = x + y + carry`, tạo nút mới chứa `total % 10`. Vì ta tạo kết quả **theo thứ tự thấp → cao**, ta **chèn vào đầu** ngay (`res = ListNode(d, res)`) để kết quả tự đúng chiều (cao ở đầu), nên **không cần đảo kết quả**.
- Vòng lặp chạy `while a or b or carry` để không bỏ sót chữ số của số dài hơn hay số nhớ cuối cùng.

Nhược điểm: `l1`, `l2` bị **đảo** (có thể đảo lại sau nếu cần), vi phạm tinh thần câu hỏi nâng cao.

In [12]:
def add_reverse(l1, l2):
    a = reverse_list(l1)
    b = reverse_list(l2)
    carry = 0
    res = None
    while a or b or carry:
        x = a.val if a else 0
        y = b.val if b else 0
        carry, digit = divmod(x + y + carry, 10)
        res = ListNode(digit, res)         # chèn đầu: chữ số cao dần ra đầu
        a = a.next if a else None
        b = b.next if b else None
    return res


for a, b, expected in examples:
    print(to_list(add_reverse(build(a), build(b))), "(kỳ vọng", expected, ")")

[7, 8, 0, 7] (kỳ vọng [7, 8, 0, 7] )
[8, 0, 7] (kỳ vọng [8, 0, 7] )
[0] (kỳ vọng [0] )


In [13]:
class SolutionReverse:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        def reverse(head):
            prev = None
            while head:
                nxt = head.next
                head.next = prev
                prev = head
                head = nxt
            return prev

        a, b = reverse(l1), reverse(l2)
        carry = 0
        res = None
        while a or b or carry:
            x = a.val if a else 0
            y = b.val if b else 0
            carry, digit = divmod(x + y + carry, 10)
            res = ListNode(digit, res)
            a = a.next if a else None
            b = b.next if b else None
        return res


print(to_list(SolutionReverse().addTwoNumbers(build([7, 2, 4, 3]), build([5, 6, 4]))))

[7, 8, 0, 7]


**Độ phức tạp (Cách 2).** Đảo hai danh sách `O(n + m)`, cộng `O(max(n, m))`: **Time `O(n + m)`**. **Space `O(1)`** phụ (không tính danh sách kết quả, độ dài `max(n, m) + 1`). Hai danh sách đầu vào bị thay đổi.

### 7.2. Cách 3 — Hai ngăn xếp + chèn vào đầu (đáp án câu hỏi nâng cao)

**Giải thích từng đoạn.**

- Duyệt `l1`, đẩy từng chữ số vào `s1`; tương tự `l2` vào `s2`. Danh sách gốc **không bị sửa**.
- `carry = 0`, `res = None`.
- `while s1 or s2 or carry:` — lấy `x = s1.pop() if s1 else 0`, `y = s2.pop() if s2 else 0`; `carry, digit = divmod(x + y + carry, 10)`; `res = ListNode(digit, res)`.
- Đỉnh ngăn xếp luôn là chữ số **thấp nhất chưa xử lý**, đúng thứ tự cộng.

Điều kiện `or carry` bảo đảm số nhớ cuối (ví dụ `5 + 5 = 10` → `[1, 0]`) không bị mất.

In [14]:
def add_stacks(l1, l2):
    s1, s2 = [], []
    while l1:
        s1.append(l1.val)
        l1 = l1.next
    while l2:
        s2.append(l2.val)
        l2 = l2.next
    carry = 0
    res = None
    while s1 or s2 or carry:
        x = s1.pop() if s1 else 0
        y = s2.pop() if s2 else 0
        carry, digit = divmod(x + y + carry, 10)
        res = ListNode(digit, res)
    return res


for a, b, expected in examples:
    print(to_list(add_stacks(build(a), build(b))), "(kỳ vọng", expected, ")")

[7, 8, 0, 7] (kỳ vọng [7, 8, 0, 7] )
[8, 0, 7] (kỳ vọng [8, 0, 7] )
[0] (kỳ vọng [0] )


In [15]:
class SolutionStacks:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        s1, s2 = [], []
        while l1:
            s1.append(l1.val)
            l1 = l1.next
        while l2:
            s2.append(l2.val)
            l2 = l2.next
        carry = 0
        res = None
        while s1 or s2 or carry:
            x = s1.pop() if s1 else 0
            y = s2.pop() if s2 else 0
            carry, digit = divmod(x + y + carry, 10)
            res = ListNode(digit, res)
        return res


print(to_list(SolutionStacks().addTwoNumbers(build([5]), build([5]))))

[1, 0]


**Độ phức tạp (Cách 3).** **Time `O(n + m)`**, **Space `O(n + m)`** cho hai ngăn xếp (không sửa đầu vào). Đây là lời giải kinh điển cho câu hỏi nâng cao.

### 7.3. Cách 4 — Căn hàng theo độ dài + đệ quy

**Giải thích từng đoạn.**

- Đếm độ dài `n1`, `n2`. Gọi `a` là danh sách **dài hơn**, `b` là ngắn hơn, `diff = |n1 - n2|`.
- Hàm đệ quy `dfs(a, b, diff)` trả về **số nhớ** đi ra từ vị trí hiện tại:
  - `a is None`: hết danh sách ⇒ số nhớ `0`.
  - Nếu `diff > 0`: nút `a` hiện tại chưa có đối tác (phần đầu của số dài). Gọi `carry = dfs(a.next, b, diff - 1)`, `total = a.val + carry`.
  - Ngược lại (đã căn hàng): `carry = dfs(a.next, b.next, 0)`, `total = a.val + b.val + carry`.
- **Sau khi** các lời gọi con trả về (tức đã xử lý các cột bên phải), ta tạo nút `total % 10` và **chèn vào đầu** danh sách kết quả; trả về `total // 10`.
- Sau lời gọi đầu tiên, nếu số nhớ còn `1` ta chèn thêm một nút `1` ở đầu.

Cách này cũng không sửa đầu vào và không cần ngăn xếp tự tạo; "ngăn xếp" là ngăn xếp lời gọi (độ sâu tối đa `max(n, m)`).

In [16]:
def add_recursive(l1, l2):
    n1, n2 = length(l1), length(l2)
    if n1 < n2:
        l1, l2 = l2, l1
    diff = abs(n1 - n2)
    head = None

    def dfs(a, b, d):
        nonlocal head
        if a is None:
            return 0
        if d > 0:
            carry = dfs(a.next, b, d - 1)
            total = a.val + carry
        else:
            carry = dfs(a.next, b.next, 0)
            total = a.val + b.val + carry
        head = ListNode(total % 10, head)
        return total // 10

    carry = dfs(l1, l2, diff)
    if carry:
        head = ListNode(carry, head)
    return head


for a, b, expected in examples:
    print(to_list(add_recursive(build(a), build(b))), "(kỳ vọng", expected, ")")

[7, 8, 0, 7] (kỳ vọng [7, 8, 0, 7] )
[8, 0, 7] (kỳ vọng [8, 0, 7] )
[0] (kỳ vọng [0] )


In [17]:
class SolutionRecursive:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        def size(node):
            n = 0
            while node:
                n += 1
                node = node.next
            return n

        n1, n2 = size(l1), size(l2)
        if n1 < n2:
            l1, l2 = l2, l1
        head = None

        def dfs(a, b, d):
            nonlocal head
            if a is None:
                return 0
            if d > 0:
                carry = dfs(a.next, b, d - 1)
                total = a.val + carry
            else:
                carry = dfs(a.next, b.next, 0)
                total = a.val + b.val + carry
            head = ListNode(total % 10, head)
            return total // 10

        carry = dfs(l1, l2, abs(n1 - n2))
        if carry:
            head = ListNode(carry, head)
        return head


print(to_list(SolutionRecursive().addTwoNumbers(build([9, 9, 9]), build([1]))))

[1, 0, 0, 0]


**Độ phức tạp (Cách 4).** Đếm độ dài `O(n + m)`, đệ quy `O(max(n, m))`: **Time `O(n + m)`**. **Space `O(max(n, m))`** cho ngăn xếp lời gọi (không có cấu trúc dữ liệu phụ nào khác). Với `n <= 100` an toàn; với danh sách rất dài (hơn ~1000) sẽ gặp `RecursionError` của Python.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái — Cách 3 (hai ngăn xếp), `l1 = [7,2,4,3]`, `l2 = [5,6,4]`

Sau khi đẩy: `s1 = [7,2,4,3]` (đỉnh ở bên phải), `s2 = [5,6,4]`.

| Vòng | `pop` từ `s1` | `pop` từ `s2` | `carry` vào | `total` | Chữ số ghi | `carry` ra | `res` sau (chèn đầu) |
|---|---|---|---|---|---|---|---|
| 1 | 3 | 4 | 0 | 7 | 7 | 0 | `[7]` |
| 2 | 4 | 6 | 0 | 10 | 0 | 1 | `[0, 7]` |
| 3 | 2 | 5 | 1 | 8 | 8 | 0 | `[8, 0, 7]` |
| 4 | 7 | (rỗng → 0) | 0 | 7 | 7 | 0 | `[7, 8, 0, 7]` |

Sau vòng 4: `s1` và `s2` rỗng, `carry = 0` ⇒ dừng; kết quả `[7, 8, 0, 7]`.

In [18]:
def trace_stacks(a, b):
    l1, l2 = build(a), build(b)
    s1, s2 = [], []
    while l1:
        s1.append(l1.val)
        l1 = l1.next
    while l2:
        s2.append(l2.val)
        l2 = l2.next
    print("s1 =", s1, "| s2 =", s2)
    carry = 0
    res = None
    step = 0
    while s1 or s2 or carry:
        step += 1
        x = s1.pop() if s1 else 0
        y = s2.pop() if s2 else 0
        total = x + y + carry
        new_carry, digit = divmod(total, 10)
        res = ListNode(digit, res)
        print(f"vòng {step}: x={x} y={y} carry vào={carry} -> total={total}, ghi {digit}, carry ra={new_carry} | res={to_list(res)}")
        carry = new_carry
    return to_list(res)


print(trace_stacks([7, 2, 4, 3], [5, 6, 4]))
print(trace_stacks([5], [5]))
print(trace_stacks([9, 9, 9], [1]))

s1 = [7, 2, 4, 3] | s2 = [5, 6, 4]
vòng 1: x=3 y=4 carry vào=0 -> total=7, ghi 7, carry ra=0 | res=[7]
vòng 2: x=4 y=6 carry vào=0 -> total=10, ghi 0, carry ra=1 | res=[0, 7]
vòng 3: x=2 y=5 carry vào=1 -> total=8, ghi 8, carry ra=0 | res=[8, 0, 7]
vòng 4: x=7 y=0 carry vào=0 -> total=7, ghi 7, carry ra=0 | res=[7, 8, 0, 7]
[7, 8, 0, 7]
s1 = [5] | s2 = [5]
vòng 1: x=5 y=5 carry vào=0 -> total=10, ghi 0, carry ra=1 | res=[0]
vòng 2: x=0 y=0 carry vào=1 -> total=1, ghi 1, carry ra=0 | res=[1, 0]
[1, 0]
s1 = [9, 9, 9] | s2 = [1]
vòng 1: x=9 y=1 carry vào=0 -> total=10, ghi 0, carry ra=1 | res=[0]
vòng 2: x=9 y=0 carry vào=1 -> total=10, ghi 0, carry ra=1 | res=[0, 0]
vòng 3: x=9 y=0 carry vào=1 -> total=10, ghi 0, carry ra=1 | res=[0, 0, 0]
vòng 4: x=0 y=0 carry vào=1 -> total=1, ghi 1, carry ra=0 | res=[1, 0, 0, 0]
[1, 0, 0, 0]


### 8.2. Bảng trạng thái — Cách 4 (đệ quy), `a = [7,2,4,3]` (dài hơn), `b = [5,6,4]`, `diff = 1`

Các lời gọi đi xuống (cột "gọi"), rồi trả về từ dưới lên (cột "trả về"):

| Độ sâu | Nút `a` | Nút `b` | `d` | Gọi | Khi quay về: `total` | Ghi | Trả về `carry` |
|---|---|---|---|---|---|---|---|
| 0 | 7 | (chưa có) | 1 | `dfs(2, 5, 0)` | 7 + 0 = 7 | 7 | 0 |
| 1 | 2 | 5 | 0 | `dfs(4, 6, 0)` | 2 + 5 + 1 = 8 | 8 | 0 |
| 2 | 4 | 6 | 0 | `dfs(3, 4, 0)` | 4 + 6 + 0 = 10 | 0 | 1 |
| 3 | 3 | 4 | 0 | `dfs(None, None, 0)` → 0 | 3 + 4 + 0 = 7 | 7 | 0 |

Thứ tự **ghi** là: độ sâu 3 (7), rồi 2 (0), rồi 1 (8), rồi 0 (7): đúng thứ tự thấp → cao, mỗi lần chèn đầu ⇒ kết quả `[7, 8, 0, 7]`.

In [19]:
def trace_recursive(a, b):
    l1, l2 = build(a), build(b)
    n1, n2 = length(l1), length(l2)
    if n1 < n2:
        l1, l2 = l2, l1
    diff = abs(n1 - n2)
    head = None
    print(f"n1={n1} n2={n2} diff={diff}")

    def dfs(x, y, d, depth):
        nonlocal head
        pad = "  " * depth
        if x is None:
            print(f"{pad}hết danh sách -> trả carry 0")
            return 0
        print(f"{pad}xuống: a={x.val} b={(y.val if d == 0 else '-')} d={d}")
        if d > 0:
            carry = dfs(x.next, y, d - 1, depth + 1)
            total = x.val + carry
        else:
            carry = dfs(x.next, y.next, 0, depth + 1)
            total = x.val + y.val + carry
        head = ListNode(total % 10, head)
        print(f"{pad}lên:   total={total} ghi {total % 10}, carry ra {total // 10} | kết quả hiện có {to_list(head)}")
        return total // 10

    carry = dfs(l1, l2, diff, 0)
    if carry:
        head = ListNode(carry, head)
        print("carry cuối =", carry, "-> chèn đầu:", to_list(head))
    return to_list(head)


print(trace_recursive([7, 2, 4, 3], [5, 6, 4]))
print(trace_recursive([9, 9], [1]))

n1=4 n2=3 diff=1
xuống: a=7 b=- d=1
  xuống: a=2 b=5 d=0
    xuống: a=4 b=6 d=0
      xuống: a=3 b=4 d=0
        hết danh sách -> trả carry 0
      lên:   total=7 ghi 7, carry ra 0 | kết quả hiện có [7]
    lên:   total=10 ghi 0, carry ra 1 | kết quả hiện có [0, 7]
  lên:   total=8 ghi 8, carry ra 0 | kết quả hiện có [8, 0, 7]
lên:   total=7 ghi 7, carry ra 0 | kết quả hiện có [7, 8, 0, 7]
[7, 8, 0, 7]
n1=2 n2=1 diff=1
xuống: a=9 b=- d=1
  xuống: a=9 b=1 d=0
    hết danh sách -> trả carry 0
  lên:   total=10 ghi 0, carry ra 1 | kết quả hiện có [0]
lên:   total=10 ghi 0, carry ra 1 | kết quả hiện có [0, 0]
carry cuối = 1 -> chèn đầu: [1, 0, 0]
[1, 0, 0]


### 8.3. Trace Cách 2 (đảo ngược)

Đảo `[7,2,4,3]` thành `[3,4,2,7]` và `[5,6,4]` thành `[4,6,5]`, rồi cộng từ đầu: `3+4=7`, `4+6=10 → 0 nhớ 1`, `2+5+1=8`, `7+0=7`: tạo `res` bằng cách chèn đầu: `[7]`, `[0,7]`, `[8,0,7]`, `[7,8,0,7]`.

In [20]:
a, b = build([7, 2, 4, 3]), build([5, 6, 4])
ra, rb = reverse_list(a), reverse_list(b)
print("sau khi đảo:", to_list(ra), to_list(rb))
carry, res = 0, None
while ra or rb or carry:
    x = ra.val if ra else 0
    y = rb.val if rb else 0
    carry, d = divmod(x + y + carry, 10)
    res = ListNode(d, res)
    ra = ra.next if ra else None
    rb = rb.next if rb else None
    print("res =", to_list(res), "carry =", carry)

sau khi đảo: [3, 4, 2, 7] [4, 6, 5]
res = [7] carry = 0
res = [0, 7] carry = 1
res = [8, 0, 7] carry = 0
res = [7, 8, 0, 7] carry = 0


## 9. Code nộp LeetCode

Ta nộp **Cách 3 (hai ngăn xếp)**: không sửa đầu vào (trả lời câu hỏi nâng cao), đơn giản, không lo giới hạn đệ quy.

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        s1, s2 = [], []
        while l1:
            s1.append(l1.val)
            l1 = l1.next
        while l2:
            s2.append(l2.val)
            l2 = l2.next
        carry = 0
        res = None
        while s1 or s2 or carry:
            x = s1.pop() if s1 else 0
            y = s2.pop() if s2 else 0
            carry, digit = divmod(x + y + carry, 10)
            res = ListNode(digit, res)
        return res
```

In [21]:
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        s1, s2 = [], []
        while l1:
            s1.append(l1.val)
            l1 = l1.next
        while l2:
            s2.append(l2.val)
            l2 = l2.next
        carry = 0
        res = None
        while s1 or s2 or carry:
            x = s1.pop() if s1 else 0
            y = s2.pop() if s2 else 0
            carry, digit = divmod(x + y + carry, 10)
            res = ListNode(digit, res)
        return res


for a, b, expected in examples:
    got = to_list(Solution().addTwoNumbers(build(a), build(b)))
    print(got)
    assert got == expected

[7, 8, 0, 7]
[8, 0, 7]
[0]


## 10. Vì sao đúng

**Phép cộng cột đúng.** Gọi các chữ số thấp → cao của hai số là `x_0, x_1, ...` và `y_0, y_1, ...` (bằng `0` khi hết). Phép cộng viết tay cho chữ số kết quả `z_k = (x_k + y_k + c_k) mod 10` và số nhớ `c_{k+1} = (x_k + y_k + c_k) div 10`, với `c_0 = 0`. Mỗi vòng của Cách 3 tính đúng một cột `k`: `pop` trả về `x_k` (đỉnh ngăn xếp là chữ số thấp nhất chưa xử lý) và `y_k`, đúng như công thức. Số nhớ `c` luôn `<= 1` (`9 + 9 + 1 = 19`).

**Số cột.** Vòng lặp chạy khi còn chữ số ở một trong hai ngăn xếp **hoặc** còn số nhớ. Hết cả hai ngăn xếp tức là mọi cột của số dài hơn đã xử lý; nếu còn `carry = 1` thì cần thêm một cột nữa (chữ số `1` dẫn đầu); sau đó `carry = 0` và dừng.

**Thứ tự kết quả.** Ta sinh các chữ số theo thứ tự thấp → cao và mỗi lần **chèn vào đầu** danh sách, nên chữ số sinh ra **sau** (cao hơn) nằm **trước** ⇒ danh sách cuối cùng có chữ số cao nhất ở đầu, đúng yêu cầu.

**Không chữ số `0` dẫn đầu.** Chữ số cuối cùng được sinh (đứng đầu) là: số nhớ cuối `1`, hoặc cột cao nhất của số dài hơn có `x_k >= 1` (vì số đó không có `0` dẫn đầu, trừ chính số `0`; với `0 + 0` ta ghi một nút `0`). Trường hợp `x_top + y + carry` có thể bằng `0` chỉ khi cả hai số là `0`, cho `[0]`.

**Cách 2** là cùng phép cộng với "ngăn xếp" là danh sách đảo; **Cách 4** là cùng phép cộng với "ngăn xếp" là ngăn xếp lời gọi, và việc căn hàng bảo đảm hai chữ số cùng cột gặp nhau (độ chênh `diff` nút đầu của số dài chỉ cộng với `0`).

## 11. Time / Space Complexity

Gọi `n`, `m` là độ dài hai danh sách.

- **Hai ngăn xếp (Cách 3):** Time `O(n + m)`, Space `O(n + m)`.
- **Đảo ngược (Cách 2):** Time `O(n + m)`, Space `O(1)` phụ; nhưng sửa đầu vào.
- **Đệ quy (Cách 4):** Time `O(n + m)`, Space `O(max(n, m))` (ngăn xếp lời gọi).
- **Đổi thành số nguyên (Cách 1):** `O(n + m)` phép toán số lớn; bỏ qua ý nghĩa bài toán.

Cả `n` và `m` tối đa `100` nên mọi cách đều tức thì. Kết quả luôn có `max(n, m)` hoặc `max(n, m) + 1` nút.

## 12. Bộ test

Ta xây danh sách từ mảng mới cho **mỗi lần gọi** (vì Cách 2 sửa đầu vào). Oracle độc lập: ghép các chữ số thành chuỗi, `int(...) + int(...)`, đổi lại thành list chữ số. Với các cách **không sửa đầu vào** ta kiểm tra thêm rằng hai danh sách gốc còn nguyên.

In [22]:
def oracle(a, b):
    total = int("".join(map(str, a))) + int("".join(map(str, b)))
    return [int(c) for c in str(total)]


impls = {
    "via_int": add_via_int,
    "reverse": add_reverse,
    "stacks": add_stacks,
    "recursive": add_recursive,
    "leetcode": lambda x, y: Solution().addTwoNumbers(x, y),
    "submit_reverse": lambda x, y: SolutionReverse().addTwoNumbers(x, y),
    "submit_stacks": lambda x, y: SolutionStacks().addTwoNumbers(x, y),
    "submit_recursive": lambda x, y: SolutionRecursive().addTwoNumbers(x, y),
}
preserve = {"via_int", "stacks", "recursive", "leetcode", "submit_stacks", "submit_recursive"}


def check_all(a, b, expected, label):
    assert oracle(a, b) == expected, (label, "oracle", oracle(a, b), expected)
    for key, fn in impls.items():
        l1, l2 = build(a), build(b)
        got = to_list(fn(l1, l2))
        assert got == expected, (label, key, a, b, got, expected)
        if key in preserve:
            assert to_list(l1) == a and to_list(l2) == b, (label, key, "đầu vào bị sửa")
    print("PASS", label, "->", expected)


for idx, (a, b, expected) in enumerate(examples, 1):
    check_all(a, b, expected, f"ví dụ {idx}")

PASS ví dụ 1 -> [7, 8, 0, 7]
PASS ví dụ 2 -> [8, 0, 7]
PASS ví dụ 3 -> [0]


### 12.1. Các trường hợp biên

In [23]:
check_all([0], [0], [0], "0 + 0")
check_all([0], [5], [5], "0 + số một chữ số")
check_all([5], [0], [5], "số một chữ số + 0")
check_all([5], [5], [1, 0], "5 + 5: số nhớ cuối")
check_all([9], [1], [1, 0], "9 + 1")
check_all([9, 9, 9], [1], [1, 0, 0, 0], "999 + 1: nhớ lan hết")
check_all([1], [9, 9, 9], [1, 0, 0, 0], "1 + 999: số ngắn đứng trước")
check_all([9, 9, 9], [9, 9, 9], [1, 9, 9, 8], "999 + 999")
check_all([1, 0, 0], [9, 9], [1, 9, 9], "100 + 99")
check_all([5, 0, 0], [5, 0, 0], [1, 0, 0, 0], "500 + 500")
check_all([1, 2, 3], [4, 5, 6], [5, 7, 9], "không nhớ")
check_all([2, 4, 3], [5, 6, 4], [8, 0, 7], "ví dụ 2")
check_all([1, 0, 0, 0, 0], [1], [1, 0, 0, 0, 1], "10000 + 1: số dài, số ngắn")
check_all([9] * 100, [9] * 100, [1] + [9] * 99 + [8], "hai số 100 chữ số 9")
check_all([9] * 100, [1], [1] + [0] * 100, "99...9 (100 chữ số) + 1 -> 101 chữ số")
check_all([1] + [0] * 99, [1] + [0] * 99, [2] + [0] * 99, "10^99 + 10^99")

PASS 0 + 0 -> [0]
PASS 0 + số một chữ số -> [5]
PASS số một chữ số + 0 -> [5]
PASS 5 + 5: số nhớ cuối -> [1, 0]
PASS 9 + 1 -> [1, 0]
PASS 999 + 1: nhớ lan hết -> [1, 0, 0, 0]
PASS 1 + 999: số ngắn đứng trước -> [1, 0, 0, 0]
PASS 999 + 999 -> [1, 9, 9, 8]
PASS 100 + 99 -> [1, 9, 9]
PASS 500 + 500 -> [1, 0, 0, 0]
PASS không nhớ -> [5, 7, 9]
PASS ví dụ 2 -> [8, 0, 7]
PASS 10000 + 1: số dài, số ngắn -> [1, 0, 0, 0, 1]
PASS hai số 100 chữ số 9 -> [1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8]
PASS 99...9 (100 chữ số) + 1 -> 101 chữ số -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi: (a) **quên số nhớ cuối** (`while s1 or s2` thiếu `or carry`); (b) **bỏ sót chữ số của số dài** (`while s1 and s2`); (c) **dựng kết quả bằng cách nối vào đuôi** nên bị ngược; (d) **căn lề trái** (cộng chữ số cùng chỉ số từ đầu thay vì cùng hàng đơn vị).

In [24]:
def wrong_no_final_carry(l1, l2):
    s1, s2 = [], []
    while l1:
        s1.append(l1.val); l1 = l1.next
    while l2:
        s2.append(l2.val); l2 = l2.next
    carry, res = 0, None
    while s1 or s2:                      # SAI: thiếu "or carry"
        x = s1.pop() if s1 else 0
        y = s2.pop() if s2 else 0
        carry, digit = divmod(x + y + carry, 10)
        res = ListNode(digit, res)
    return res


def wrong_and(l1, l2):
    s1, s2 = [], []
    while l1:
        s1.append(l1.val); l1 = l1.next
    while l2:
        s2.append(l2.val); l2 = l2.next
    carry, res = 0, None
    while s1 and s2:                     # SAI: dừng khi một ngăn xếp hết
        carry, digit = divmod(s1.pop() + s2.pop() + carry, 10)
        res = ListNode(digit, res)
    if carry:
        res = ListNode(carry, res)
    return res


def wrong_append_tail(l1, l2):
    s1, s2 = [], []
    while l1:
        s1.append(l1.val); l1 = l1.next
    while l2:
        s2.append(l2.val); l2 = l2.next
    carry = 0
    dummy = ListNode()
    tail = dummy
    while s1 or s2 or carry:
        x = s1.pop() if s1 else 0
        y = s2.pop() if s2 else 0
        carry, digit = divmod(x + y + carry, 10)
        tail.next = ListNode(digit)      # SAI: nối vào đuôi -> chữ số thấp ở đầu
        tail = tail.next
    return dummy.next


def wrong_left_aligned(l1, l2):
    a, b = to_list(l1), to_list(l2)
    m = max(len(a), len(b))
    a += [0] * (m - len(a))              # SAI: đệm số 0 vào cuối (căn lề trái)
    b += [0] * (m - len(b))
    total = int("".join(map(str, a))) + int("".join(map(str, b)))
    return build([int(c) for c in str(total)])


cases = [
    ("quên carry cuối", wrong_no_final_carry, [5], [5]),
    ("bỏ chữ số số dài", wrong_and, [7, 2, 4, 3], [5, 6, 4]),
    ("nối vào đuôi", wrong_append_tail, [7, 2, 4, 3], [5, 6, 4]),
    ("căn lề trái", wrong_left_aligned, [7, 2, 4, 3], [5, 6, 4]),
]
for name, fn, a, b in cases:
    got = to_list(fn(build(a), build(b)))
    print(f"{name:>18}: sai = {got} | đúng = {oracle(a, b)}")
    assert got != oracle(a, b)

   quên carry cuối: sai = [0] | đúng = [1, 0]
  bỏ chữ số số dài: sai = [8, 0, 7] | đúng = [7, 8, 0, 7]
      nối vào đuôi: sai = [7, 0, 8, 7] | đúng = [7, 8, 0, 7]
       căn lề trái: sai = [1, 2, 8, 8, 3] | đúng = [7, 8, 0, 7]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh hai số **không có chữ số `0` dẫn đầu** (trừ số `0`), độ dài `1..12` và thỉnh thoảng cực đoan: toàn chữ số `9` (nhớ lan), hoặc độ dài rất chênh nhau. Đối chiếu mọi cách với oracle và kiểm tra các cách "giữ đầu vào" thật sự không sửa đầu vào.

In [25]:
def rand_number(rng, length):
    if length == 1:
        return [rng.randint(0, 9)]
    digits = [rng.randint(1, 9)] + [rng.randint(0, 9) for _ in range(length - 1)]
    return digits


rng = random.Random(445)
carry_chains = 0
for trial in range(3000):
    n = rng.randint(1, 12)
    m = rng.randint(1, 12)
    if rng.random() < 0.1:
        a = [9] * n
    else:
        a = rand_number(rng, n)
    if rng.random() < 0.1:
        b = [9] * m
    else:
        b = rand_number(rng, m)
    expected = oracle(a, b)
    if len(expected) > max(len(a), len(b)):
        carry_chains += 1
    for key, fn in impls.items():
        l1, l2 = build(a), build(b)
        assert to_list(fn(l1, l2)) == expected, (key, a, b)
        if key in preserve:
            assert to_list(l1) == a and to_list(l2) == b, (key, "sửa đầu vào")
print("3000 test OK; số ca tổng dài hơn cả hai số:", carry_chains)

for trial in range(300):
    a = rand_number(rng, rng.randint(1, 100))
    b = rand_number(rng, rng.randint(1, 100))
    expected = oracle(a, b)
    for key, fn in impls.items():
        assert to_list(fn(build(a), build(b))) == expected, (key, "dài")
print("300 test độ dài đến 100 OK")

3000 test OK; số ca tổng dài hơn cả hai số: 492


300 test độ dài đến 100 OK


### 12.4. Danh sách rất dài (ngoài ràng buộc): chọn đúng cách

Ràng buộc chỉ tới `100` nút. Nếu danh sách dài hàng nghìn nút, cách **đệ quy** (độ sâu đệ quy = độ dài) vượt giới hạn mặc định của Python (`sys.getrecursionlimit()` thường là `1000`) và sinh `RecursionError`; các cách lặp (ngăn xếp, đảo ngược, số nguyên) vẫn ổn. Ta bắt lỗi để minh họa, không để làm hỏng notebook. (Ta dùng cỡ 3000 chữ số vì Python giới hạn đổi số nguyên ↔ chuỗi ở khoảng 4300 chữ số mặc định, ảnh hưởng đến oracle và Cách 1.)

In [26]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
rng = random.Random(4450)
a = rand_number(rng, 3000)
b = rand_number(rng, 2500)
expected = oracle(a, b)
for key in ["stacks", "reverse", "via_int"]:
    t0 = time.perf_counter()
    assert to_list(impls[key](build(a), build(b))) == expected, key
    print(f"{key:>10}: đúng với 3000 chữ số, {1000 * (time.perf_counter() - t0):.0f} ms")
try:
    add_recursive(build(a), build(b))
    print("recursive: không lỗi (giới hạn đệ quy đã đủ)")
except RecursionError:
    print("recursive: RecursionError với 3000 nút (đúng như dự đoán; với n <= 100 thì an toàn)")

giới hạn đệ quy mặc định: 1000
    stacks: đúng với 3000 chữ số, 2 ms
   reverse: đúng với 3000 chữ số, 1 ms
   via_int: đúng với 3000 chữ số, 4 ms
recursive: RecursionError với 3000 nút (đúng như dự đoán; với n <= 100 thì an toàn)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên số nhớ cuối cùng

**Code sai:** `while s1 or s2:` (thiếu `or carry`).

**Vì sao sai:** `5 + 5 = 10` — sau khi xử lý hai chữ số, `carry = 1` còn dư nhưng vòng lặp đã dừng, ta mất chữ số `1` dẫn đầu và trả `[0]` thay vì `[1, 0]`. Tương tự `999 + 1` trả `[0, 0, 0]`.

**Code đúng:** `while s1 or s2 or carry:`.

In [27]:
print("sai :", to_list(wrong_no_final_carry(build([5]), build([5]))), to_list(wrong_no_final_carry(build([9, 9, 9]), build([1]))))
print("đúng:", to_list(Solution().addTwoNumbers(build([5]), build([5]))), to_list(Solution().addTwoNumbers(build([9, 9, 9]), build([1]))))

sai : [0] [0, 0, 0]
đúng: [1, 0] [1, 0, 0, 0]


### Lỗi 2: Dừng khi một danh sách hết, bỏ sót chữ số của số dài hơn

**Code sai:** `while s1 and s2:`.

**Vì sao sai:** số dài hơn còn các chữ số cao chưa xử lý. Ở `7243 + 564`, sau ba cột ta dừng và mất chữ số `7` đầu, kết quả `[8, 0, 7]` thay vì `[7, 8, 0, 7]`.

**Code đúng:** dùng `or` và lấy `0` cho ngăn xếp đã hết (`x = s1.pop() if s1 else 0`).

In [28]:
print("sai :", to_list(wrong_and(build([7, 2, 4, 3]), build([5, 6, 4]))))
print("đúng:", to_list(Solution().addTwoNumbers(build([7, 2, 4, 3]), build([5, 6, 4]))))

sai : [8, 0, 7]
đúng: [7, 8, 0, 7]


### Lỗi 3: Dựng kết quả bằng cách nối vào đuôi

**Code sai:** `tail.next = ListNode(digit); tail = tail.next` (như bài Add Two Numbers I).

**Vì sao sai:** ta sinh chữ số theo thứ tự **thấp → cao**, nối đuôi thì chữ số thấp nằm đầu ⇒ kết quả bị **đảo ngược** (`[7, 0, 8, 7]` thay vì `[7, 8, 0, 7]`). Ở bài này cần chèn vào **đầu**: `res = ListNode(digit, res)`.

In [29]:
print("sai :", to_list(wrong_append_tail(build([7, 2, 4, 3]), build([5, 6, 4]))))
print("đúng:", to_list(Solution().addTwoNumbers(build([7, 2, 4, 3]), build([5, 6, 4]))))

sai : [7, 0, 8, 7]
đúng: [7, 8, 0, 7]


### Lỗi 4: Căn lề trái (cộng cùng chỉ số từ đầu)

**Code sai:** cộng `l1[i] + l2[i]` theo cùng chỉ số tính từ đầu danh sách, hoặc đệm số `0` vào **cuối** số ngắn.

**Vì sao sai:** hai số cộng với nhau phải thẳng hàng theo **hàng đơn vị** (căn lề phải). Đệm `0` vào cuối là nhân số ngắn với `10^k`: `[5,6,4]` thành `[5,6,4,0]` = `5640` thay vì `564`.

**Code đúng:** ghép từ cuối (ngăn xếp), hoặc căn hàng bằng độ chênh `diff` ở đầu số dài.

In [30]:
print("sai :", to_list(wrong_left_aligned(build([7, 2, 4, 3]), build([5, 6, 4]))), "(= 7243 + 5640)")
print("đúng:", to_list(Solution().addTwoNumbers(build([7, 2, 4, 3]), build([5, 6, 4]))), "(= 7243 + 564)")

sai : [1, 2, 8, 8, 3] (= 7243 + 5640)
đúng: [7, 8, 0, 7] (= 7243 + 564)


### Lỗi 5: Quên rằng cách đảo ngược làm hỏng đầu vào

**Code sai (dễ miss):** đảo `l1`, `l2` rồi cộng, sau đó dùng lại `l1`/`l2` ở chỗ khác.

**Vì sao sai:** sau `reverse(l1)`, biến `l1` vẫn trỏ nút **đầu cũ**, nay là **đuôi** (`l1.next is None`), nên đọc `l1` chỉ thấy một nút. Câu hỏi nâng cao yêu cầu không sửa đầu vào chính vì điều này. Hãy dùng ngăn xếp hoặc đệ quy, hoặc đảo lại sau khi xong.

In [31]:
l1 = build([7, 2, 4, 3])
l2 = build([5, 6, 4])
add_reverse(l1, l2)
print("l1 sau khi gọi add_reverse:", to_list(l1), "(chỉ còn nút đầu cũ, vì danh sách đã bị đảo)")
l1 = build([7, 2, 4, 3])
l2 = build([5, 6, 4])
Solution().addTwoNumbers(l1, l2)
print("l1 sau khi gọi Solution:   ", to_list(l1), "(nguyên vẹn)")

l1 sau khi gọi add_reverse: [7] (chỉ còn nút đầu cũ, vì danh sách đã bị đảo)
l1 sau khi gọi Solution:    [7, 2, 4, 3] (nguyên vẹn)


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận dạng:** "cộng các số được lưu theo thứ tự cao → thấp trong danh sách liên kết" và "không đảo danh sách" ⇒ cần đi theo thứ tự **ngược** với chiều duyệt: **ngăn xếp** (hoặc đệ quy). Kết quả dựng bằng **chèn vào đầu**.

Bảng nhắc:

| Tình huống | Công cụ |
|---|---|
| Cần xử lý từ cuối về đầu trên danh sách liên kết đơn | Ngăn xếp / đệ quy / đảo danh sách |
| Kết quả sinh theo thứ tự ngược, cần xuôi | Chèn vào đầu `head = ListNode(v, head)` |
| Số nhớ | `carry, digit = divmod(total, 10)`; vòng lặp `while ... or carry` |
| Hai danh sách khác độ dài | Lấy `0` khi một bên hết, hoặc căn hàng bằng `diff` |
| Số cực lớn không nằm vừa `long` | Làm từng chữ số, đừng đổi thành số nguyên |

Bài cùng họ: **2 Add Two Numbers**, **369 Plus One Linked List**, **989 Add to Array-Form of Integer**, **67 Add Binary**, **43 Multiply Strings**, **206 Reverse Linked List**.

## 15. Biến thể

### 15.1. Add Two Numbers (I): chữ số **thấp** nằm ở đầu

Khi chữ số thấp ở đầu, ta đi thẳng từ đầu mà không cần đảo gì; chỉ cần **nối vào đuôi**.

In [32]:
def add_two_numbers_I(l1, l2):
    dummy = ListNode()
    tail = dummy
    carry = 0
    while l1 or l2 or carry:
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0
        carry, d = divmod(x + y + carry, 10)
        tail.next = ListNode(d)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next


# 342 + 465 = 807 với chữ số thấp ở đầu: [2,4,3] + [5,6,4] = [7,0,8]
assert to_list(add_two_numbers_I(build([2, 4, 3]), build([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers_I(build([9, 9]), build([1]))) == [0, 0, 1]
print(to_list(add_two_numbers_I(build([2, 4, 3]), build([5, 6, 4]))))

[7, 0, 8]


### 15.2. Cộng một (Plus One Linked List, LeetCode 369)

Cho một số lưu cao → thấp, hãy cộng `1`. Mẹo không cần đảo/ngăn xếp: tìm **nút cuối cùng khác `9`** — mọi nút `9` sau nó thành `0` và nút đó tăng thêm `1`. Nếu không có nút nào khác `9` thì chèn thêm nút `1` ở đầu.

In [33]:
def plus_one(head):
    sentinel = ListNode(0, head)            # nút "0" dẫn đầu để gom trường hợp toàn số 9
    not_nine = sentinel
    node = head
    while node:
        if node.val != 9:
            not_nine = node
        node = node.next
    not_nine.val += 1
    node = not_nine.next
    while node:
        node.val = 0
        node = node.next
    return sentinel if sentinel.val else sentinel.next


assert to_list(plus_one(build([1, 2, 3]))) == [1, 2, 4]
assert to_list(plus_one(build([1, 9, 9]))) == [2, 0, 0]
assert to_list(plus_one(build([9, 9]))) == [1, 0, 0]
assert to_list(plus_one(build([0]))) == [1]
print(to_list(plus_one(build([1, 9, 9]))))

[2, 0, 0]


### 15.3. Cộng nhiều danh sách (`k` số)

Cộng lần lượt kết quả với danh sách tiếp theo bằng chính `Solution().addTwoNumbers` (độ dài kết quả tăng dần theo `log(k)`).

In [34]:
def add_many(lists):
    acc = lists[0]
    for nxt in lists[1:]:
        acc = Solution().addTwoNumbers(acc, nxt)
    return acc


nums_list = [[9, 9], [1], [5, 5, 5], [0]]
got = to_list(add_many([build(x) for x in nums_list]))
expected = [int(c) for c in str(sum(int("".join(map(str, x))) for x in nums_list))]
print(got, expected)
assert got == expected

[6, 5, 5] [6, 5, 5]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đổi thành số nguyên | `int` rồi cộng rồi tách chữ số | `O(n + m)` (số lớn) | `O(n + m)` | Chỉ Python; làm oracle |
| 2. Đảo ngược hai danh sách | Cộng như bài I rồi chèn đầu | `O(n + m)` | `O(1)` phụ | Khi được phép sửa đầu vào |
| 3. Hai ngăn xếp | `pop` ra từ chữ số thấp, chèn đầu | `O(n + m)` | `O(n + m)` | **Lời giải nộp**; trả lời follow-up |
| 4. Căn hàng + đệ quy | `dfs` đi xuống, cộng khi quay về | `O(n + m)` | `O(max(n, m))` (lời gọi) | Khi thích đệ quy; `n` nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `divmod(17, 10)`, `to_list(Solution().addTwoNumbers(build([9]), build([9])))`, `to_list(Solution().addTwoNumbers(build([1, 0]), build([9])))`, `to_list(reverse_list(build([1, 2, 3])))`, `length(build([4, 4, 4, 4]))`.

**Bài 2 (tính tay).** Với `l1 = [3, 9, 9]` và `l2 = [8, 1]`: lập bảng cộng theo cột (ngăn xếp `pop`, `carry` vào, `total`, chữ số ghi, `carry` ra, `res` hiện có) và nêu kết quả.

**Bài 3 (điền TODO).** Hoàn thành `add_todo(l1, l2)` theo Cách 3 (hai ngăn xếp).

In [35]:
def add_todo(l1, l2):
    s1, s2 = [], []
    # TODO 1: đẩy các val của l1 vào s1, của l2 vào s2
    carry = 0
    res = None
    # TODO 2: while s1 or s2 or carry: pop (hoặc 0), divmod, chèn đầu: res = ListNode(digit, res)
    return res

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `add_reverse_restore(l1, l2)` theo Cách 2 nhưng **đảo lại** hai danh sách đầu vào sau khi cộng xong để chúng trở về như cũ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `plus_one_recursive(head)` (cộng thêm 1) bằng đệ quy: hàm đệ quy trả về số nhớ. (b) Viết `compare_numbers(l1, l2)` trả `-1`, `0` hoặc `1` khi so sánh hai số (cao → thấp, không có `0` dẫn đầu) **mà không đổi sang số nguyên**: so độ dài trước, nếu bằng nhau thì so chữ số đầu tiên khác nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(1, 7)`, `[1, 8]`, `[1, 9]` (vì `10 + 9 = 19`), `[3, 2, 1]`, `4`. Bài 3: dùng `while l1: s1.append(l1.val); l1 = l1.next`.
- **Mức 2 (vừa):** Bài 2: `s1 = [3,9,9]`, `s2 = [8,1]`: `9+1=10`→ghi 0 nhớ 1; `9+8+1=18`→ghi 8 nhớ 1; `3+0+1=4`→ghi 4 ⇒ `[4, 8, 0]` (399 + 81 = 480). Bài 4: gọi `reverse_list` một lần nữa trên các đầu **mới** (`a`, `b` sau khi đảo) — nhớ giữ lại tham chiếu đến đầu đã đảo.
- **Mức 3 (sâu):** Bài 5(a): `dfs(node)` trả về số nhớ: `carry = dfs(node.next) if node.next else 1` (cộng 1 ở nút cuối), `total = node.val + carry`, `node.val = total % 10`, trả `total // 10`; nếu còn nhớ ở đầu thì chèn nút `1`. 5(b): `n1 != n2` thì `n1 > n2` quyết định; nếu bằng nhau duyệt song song, tại chữ số khác nhau đầu tiên so sánh.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [36]:
# Bài 1
assert divmod(17, 10) == (1, 7)
assert to_list(Solution().addTwoNumbers(build([9]), build([9]))) == [1, 8]
assert to_list(Solution().addTwoNumbers(build([1, 0]), build([9]))) == [1, 9]
assert to_list(reverse_list(build([1, 2, 3]))) == [3, 2, 1]
assert length(build([4, 4, 4, 4])) == 4

# Bài 2
assert to_list(Solution().addTwoNumbers(build([3, 9, 9]), build([8, 1]))) == [4, 8, 0]


# Bài 3
def add_todo_ans(l1, l2):
    s1, s2 = [], []
    while l1:
        s1.append(l1.val)
        l1 = l1.next
    while l2:
        s2.append(l2.val)
        l2 = l2.next
    carry = 0
    res = None
    while s1 or s2 or carry:
        x = s1.pop() if s1 else 0
        y = s2.pop() if s2 else 0
        carry, d = divmod(x + y + carry, 10)
        res = ListNode(d, res)
    return res


for a, b, expected in examples:
    assert to_list(add_todo_ans(build(a), build(b))) == expected


# Bài 4
def add_reverse_restore(l1, l2):
    a, b = reverse_list(l1), reverse_list(l2)
    ha, hb = a, b                            # giữ đầu đã đảo để đảo lại
    carry, res = 0, None
    while a or b or carry:
        x = a.val if a else 0
        y = b.val if b else 0
        carry, d = divmod(x + y + carry, 10)
        res = ListNode(d, res)
        a = a.next if a else None
        b = b.next if b else None
    reverse_list(ha)                         # đảo lại: l1 là đầu của danh sách khôi phục
    reverse_list(hb)
    return res


l1, l2 = build([7, 2, 4, 3]), build([5, 6, 4])
assert to_list(add_reverse_restore(l1, l2)) == [7, 8, 0, 7]
assert to_list(l1) == [7, 2, 4, 3] and to_list(l2) == [5, 6, 4]


# Bài 5
def plus_one_recursive(head):
    def dfs(node):
        if node is None:
            return 1                         # "cộng 1" bắt đầu ở sau nút cuối
        carry = dfs(node.next)
        total = node.val + carry
        node.val = total % 10
        return total // 10

    carry = dfs(head)
    return ListNode(1, head) if carry else head


assert to_list(plus_one_recursive(build([1, 2, 9]))) == [1, 3, 0]
assert to_list(plus_one_recursive(build([9, 9]))) == [1, 0, 0]
assert to_list(plus_one_recursive(build([0]))) == [1]


def compare_numbers(l1, l2):
    n1, n2 = length(l1), length(l2)
    if n1 != n2:
        return -1 if n1 < n2 else 1
    while l1:
        if l1.val != l2.val:
            return -1 if l1.val < l2.val else 1
        l1, l2 = l1.next, l2.next
    return 0


assert compare_numbers(build([1, 2, 3]), build([1, 2, 3])) == 0
assert compare_numbers(build([1, 2, 3]), build([1, 3, 0])) == -1
assert compare_numbers(build([9]), build([1, 0])) == -1
assert compare_numbers(build([2, 0, 0]), build([1, 9, 9])) == 1
rng = random.Random(4451)
for _ in range(300):
    a = rand_number(rng, rng.randint(1, 8))
    b = rand_number(rng, rng.randint(1, 8))
    x, y = int("".join(map(str, a))), int("".join(map(str, b)))
    assert compare_numbers(build(a), build(b)) == (x > y) - (x < y)
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cộng hai số lưu cao → thấp, không đảo đầu vào)

```python
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        s1, s2 = [], []
        while l1:
            s1.append(l1.val)
            l1 = l1.next
        while l2:
            s2.append(l2.val)
            l2 = l2.next
        carry = 0
        res = None
        while s1 or s2 or carry:                 # còn chữ số hoặc còn số nhớ
            x = s1.pop() if s1 else 0
            y = s2.pop() if s2 else 0
            carry, digit = divmod(x + y + carry, 10)
            res = ListNode(digit, res)           # chèn ĐẦU: chữ số cao dần ra trước
        return res
```

**Checklist**

- [ ] Vòng lặp có `or carry` (không mất số nhớ cuối)?
- [ ] Dùng `or` (không phải `and`) để không bỏ sót chữ số của số dài hơn, và lấy `0` khi hết?
- [ ] Kết quả dựng bằng **chèn vào đầu** (`ListNode(digit, res)`), không nối đuôi?
- [ ] Đã tránh sửa đầu vào (không đảo danh sách) cho câu hỏi nâng cao?
- [ ] Đã thử `[0]+[0]`, `[5]+[5]`, `[9,9,9]+[1]`, `[1]+[9,9,9]`, hai số 100 chữ số?
- [ ] Skeleton đúng `def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]`? (ListNode do LeetCode cung cấp sẵn; ở máy bạn phải tự định nghĩa.)